# Hacker News, npm & PyPI Developer Signals

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mamoo85/open-data-actor-notebooks/blob/main/notebooks/developer-signals.ipynb)

**What:** Track Hacker News mentions, Show HN and Launch HN posts, plus npm and PyPI package versions, repo links and download counts, in one dataset. Keyless public APIs. From $0.001 per item.

**Why:** run the [`developer-signals`](https://apify.com/transparent_meteorite/developer-signals) Apify actor from Python and analyse the result as a pandas DataFrame, with no scraping code to maintain.

> **Disclosure:** I am the author of this actor. It is pay-per-result; the free plan credit on a new Apify account covers a trial run like this one (this notebook uses a very small sample input).

**Privacy:** Post author handles are dropped from the table.

## 1. Setup

In [1]:
!pip install -q apify-client pandas

import os, getpass
try:
    from google.colab import userdata
    APIFY_TOKEN = userdata.get('APIFY_TOKEN')          # Colab: add it under the key icon (Secrets)
except Exception:
    APIFY_TOKEN = os.environ.get('APIFY_TOKEN') or getpass.getpass('Apify API token: ')
assert APIFY_TOKEN, 'An Apify API token is required (a free account is enough).'
print('Token loaded.')

Token loaded.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Input
A deliberately small sample so the run is cheap and quick. See the [actor page](https://apify.com/transparent_meteorite/developer-signals) for every option.

In [2]:
run_input = {   'modes': ['hnSearch', 'showHn', 'npmPackages', 'pypiPackages'],
    'hnKeywords': ['apify'],
    'sinceDate': '90 days',
    'maxItemsPerQuery': 3,
    'npmPackages': ['crawlee'],
    'pypiPackages': ['apify-client']}

## 3. Run the actor and read the dataset

In [3]:
from apify_client import ApifyClient

client = ApifyClient(APIFY_TOKEN)
run = client.actor('transparent_meteorite/developer-signals').call(run_input=run_input)
# apify-client v2 returns a dict, v3 a Run object; handle both
get = (lambda k, a: run[k]) if isinstance(run, dict) else (lambda k, a: getattr(run, a))
items = list(client.dataset(get('defaultDatasetId', 'default_dataset_id')).iterate_items())
print('Run status:', get('status', 'status'), '| items:', len(items))

[apify.developer-signals runId:VHI9nlnTEt2kFfePc] -> Status: RUNNING, Message: 


[apify.developer-signals runId:VHI9nlnTEt2kFfePc] -> 2026-10-13T22:17:43.888Z ACTOR: Pulling container image of build ItViP0QenLkXvCu3O from registry.


[apify.developer-signals runId:VHI9nlnTEt2kFfePc] -> 2026-10-13T22:17:43.890Z ACTOR: Creating container.


[apify.developer-signals runId:VHI9nlnTEt2kFfePc] -> 2026-10-13T22:17:43.958Z ACTOR: Starting container.


[apify.developer-signals runId:VHI9nlnTEt2kFfePc] -> 2026-10-13T22:17:43.960Z ACTOR: Running under "LIMITED_PERMISSIONS".


...

[apify.developer-signals runId:VHI9nlnTEt2kFfePc] -> 2026-10-13T22:17:46.242Z Done in 2s: 4 HN rows, 2 packages (0 not found), 9 HTTP requests. Warnings: 0


[apify.developer-signals runId:VHI9nlnTEt2kFfePc] -> 2026-10-13T22:17:46.317Z INFO  [Status message]: Done: 6 items


[apify.developer-signals runId:VHI9nlnTEt2kFfePc] -> 2026-10-13T22:17:46.347Z Thanks for using Developer Signals. Found a bug or need a field? Open an issue on the Issues tab (https://console.apify.com/actors/NoQcagaAmd4pldubh/issues) with the run URL and we will reply there.


[apify.developer-signals runId:VHI9nlnTEt2kFfePc] -> Status: SUCCEEDED, Message: Done: 6 items


Run status: SUCCEEDED | items: 6


## 4. Tabulate

Post author handles are dropped from the table.

In [4]:
import pandas as pd
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 200)

df = pd.DataFrame(items)

# Privacy: remove contact / personal-data columns before anything is displayed
DROP = ["email", "phone", "author", "registeredagent", "contractor", "officer", "linkedin", "facebook", "instagram", "youtube", "tiktok", "socials", "twitter"]
KEEP = ["emailCount", "phoneCount", "emailProvider"]
df = df[[c for c in df.columns if c in KEEP or not any(p in c.lower() for p in DROP)]]
SHOW = [c for c in ["source", "recordType", "kind", "title", "domain", "createdAt", "points", "numComments", "packageName", "latestVersion", "downloadsLastMonth"] if c in df.columns]
df[SHOW]

,source,recordType,kind,title,domain,createdAt,points,numComments,packageName,latestVersion,downloadsLastMonth
0,hackernews,story,showHn,"Show HN:APIFlow Studio – Open-source, offline-first Postman altern...",apiflowstudio.onrender.com,2026-10-09T05:40:43Z,3.0,1.0,NaN,NaN,NaN
1,hackernews,story,showHn,"Show HN: Vietnam Tender Intelligence – gov procurement data, AI-ag...",apify.com,2026-10-01T04:50:24Z,2.0,0.0,NaN,NaN,NaN
2,hackernews,story,story,"I built 13 small Apify Actors this month, mostly AI-grounded",apify.com,2026-09-22T12:41:09Z,1.0,0.0,NaN,NaN,NaN
3,hackernews,story,showHn,Show HN: MCP Security Auditor – Zero-execution static scanner for ...,apify.com,2026-09-04T11:19:48Z,2.0,0.0,NaN,NaN,NaN
4,npm,package,NaN,crawlee,NaN,NaN,NaN,NaN,crawlee,3.18.2,704236.0
5,pypi,package,NaN,apify-client,NaN,NaN,NaN,NaN,apify-client,3.3.0,2274981.0


## 5. A quick analysis

In [5]:
pk=df[df['recordType']=='package'].copy()
pk['downloadsLastMonth']=pd.to_numeric(pk['downloadsLastMonth'],errors='coerce')
print('Package downloads, last month:')
print(pk[['packageName','downloadsLastMonth']].sort_values('downloadsLastMonth',ascending=False).to_string(index=False))
st=df[df['recordType']=='story'].copy()
st['points']=pd.to_numeric(st['points'],errors='coerce')
st.sort_values('points',ascending=False)[['title','points','numComments']].head(5)

Package downloads, last month:
 packageName  downloadsLastMonth
apify-client           2274981.0
     crawlee            704236.0


,title,points,numComments
0,"Show HN:APIFlow Studio – Open-source, offline-first Postman altern...",3.0,1.0
1,"Show HN: Vietnam Tender Intelligence – gov procurement data, AI-ag...",2.0,0.0
3,Show HN: MCP Security Auditor – Zero-execution static scanner for ...,2.0,0.0
2,"I built 13 small Apify Actors this month, mostly AI-grounded",1.0,0.0


## Next steps

- Raise the limits in the input cell to run a larger sample.
- Schedule the actor or call it from your own pipeline: [developer-signals on the Apify Store](https://apify.com/transparent_meteorite/developer-signals).
- Check the Store page for the current pay-per-result price before scaling up.

Generated by `tools/generate.py`.